# Sumativa 2 · Validación mediante remuestreo y simulación

**Universidad Andrés Bello · Magíster en Ciencia de Datos e Inteligencia Artificial**  
**Estadística Computacional (MCDI501) · Grupo 7 · Octubre de 2026**  
Felipe Palma Barrientos · Vicente Aguila Rojas · Cristian Grandon Grandon · Jorge Gutierrez Jaramillo

**Pregunta:** ¿cuáles de los resultados de la Sumativa 1 se mantienen al repetir el análisis mediante remuestreo y cambiar algunos supuestos?

**Objetivo:** validar las tres medias, dos pruebas de hipótesis y tres correlaciones de S1; simular la diferencia de peso entre estados de entrega; identificar resultados confiables y resultados que requieren cautela para S3.

### Cómo utilizar este notebook

1. Guardar este archivo como **`notebooks/sumativa_2.ipynb`**, junto a `sumativa_1.ipynb`.
2. Abrir la carpeta del repositorio en VS Code o Jupyter y seleccionar el mismo entorno de S1, con NumPy, pandas, SciPy y Matplotlib. El código usa `rng=` de SciPy, disponible desde SciPy 1.15; S1 declara SciPy 1.17.
3. Ejecutar **Restart Kernel → Run All**. Las simulaciones pueden tardar varios minutos. Mantener **10.000** repeticiones para la entrega; `LOTE` controla la memoria.
4. Revisar las conclusiones y el archivo **`data/processed/resultados_validos_s2.json`**, creado por la última celda de exportación.

Todo el código nuevo está aquí: no requiere nuevos módulos ni nuevas carpetas. Los datos de S1 se leen sin modificarlos. Las figuras y tablas quedan integradas en el notebook. Reejecutar actualiza únicamente el JSON de S2.

### Índice y correspondencia con los criterios solicitados

| Sección | Contenido | Criterio / máximo |
|---|---|---:|
| 1–2 | Configuración, datos y reproducción de S1 | Trazabilidad transversal |
| 3 | Bootstrap percentil y BCa de tres medias | Bootstrap |
| 4 | Permutación del chi-cuadrado de S1 | Permutación |
| 5 | Bootstrap de tres correlaciones de S1 | Correlaciones |
| 6 | Monte Carlo y convergencia | Simulación |
| 7 | Influencia individual, extremos y supuestos | Robustez |
| 8–9 | Conclusiones, resultados para S3 y controles | Documentación |
**Arquitectura modular (S2).** Los cálculos residen en `src/s2_*.py`; `SimuladorMonteCarlo` implementa los ensayos de forma orientada a objetos. Este notebook coordina resultados, visualizaciones y discusión. El CSV de S1 no se modifica.


## 1. Configuración y datos

Se utiliza la versión **MCAR de S1**: 10.999 envíos, con 1.100 faltantes artificiales en peso y 1.100 en costo. Se excluyen faltantes por variable o por pares; no se imputan ni se restauran los valores ocultados.

`0 = a tiempo`; `1 = fuera de plazo`. El peso se expresa en gramos; el costo, en USD según S1. La unidad del descuento no está documentada: **no se interpreta como porcentaje**.

Cada análisis usa una semilla distinta y fija. Así, ejecutar de nuevo una sección conserva sus resultados. Las cifras de S1 se verifican antes de simular.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import subprocess
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

B = 10_000                 # Remuestras y permutaciones, por análisis.
M = 10_000                 # Iteraciones Monte Carlo, por escenario.
LOTE = 50                  # Menor lote = menor uso de memoria.
ALFA = 0.05
SEMILLAS = {
    "medias": [101, 102, 103], "permutacion": 201, "permutacion_peso": 202,
    "correlaciones": [301, 302, 303], "lognormal": 401,
    "gamma": 402, "diferencia_bootstrap": 403,
    "descuento_filtrado": 501, "correlacion_filtrada": 502,
    "escenario_inferior": 601, "escenario_superior": 602,
    "robustez_recortadas": 603,
}
COMMIT_S1 = "0c7ffaca4e45525ab434f6c9305128221e4ef1c4"
REPO_URL = "https://github.com/jorge170102/ESTAD-STICA-COMPUTACIONAL---Grupo-7--MCDI501.202682.1914.EL.ON"
HASH_DATOS_S1 = "1a7f8a5c4da839e481ff2dc93e30c7f64205407a2bd62ae94be4265027ca0714"

RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/processed/ecommerce_mcar.csv").is_file()
             and (p / "notebooks/sumativa_1.ipynb").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra este notebook dentro del repositorio, con los archivos de S1.")

# Incorporar la raíz del proyecto a las rutas de importación
import sys

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RUTA_DATOS = RAIZ / "data/processed/ecommerce_mcar.csv"
RUTA_SALIDA = RAIZ / "data/processed/resultados_validos_s2.json"
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

# Normalizar solo saltos de línea permite verificar el CSV también en Windows.
hash_datos = hashlib.sha256(RUTA_DATOS.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
assert hash_datos == HASH_DATOS_S1, "El CSV cambió: revisar la línea base de S1 antes de continuar."
df = pd.read_csv(RUTA_DATOS)
OBJETIVO = "Reached.on.Time_Y.N"
VARIABLES = ["Weight_in_gms", "Cost_of_the_Product", "Discount_offered"]
NOMBRES = dict(zip(VARIABLES, ["Peso (g)", "Costo (USD)", "Descuento (unidad no indicada)"]))

assert df.shape == (10_999, 12)
assert df.ID.notna().all() and df.ID.is_unique
assert df[OBJETIVO].isin([0, 1]).all()
assert df["Mode_of_Shipment"].isin(["Flight", "Road", "Ship"]).all()
assert df["Weight_in_gms"].isna().sum() == 1_100
assert df["Cost_of_the_Product"].isna().sum() == 1_100
assert df["Discount_offered"].notna().all()
for variable in VARIABLES:
    assert np.isfinite(df[variable].dropna()).all()

VERSIONES = {"Python": platform.python_version(), "NumPy": np.__version__,
            "pandas": pd.__version__, "SciPy": scipy.__version__,
            "Matplotlib": matplotlib.__version__}
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.precision", 4)
AZUL, NARANJA, VERDE = "#23648A", "#CF7B28", "#2F826B"
display(pd.Series(VERSIONES, name="Versión ejecutada").to_frame())
print(f"Datos verificados: {len(df):,} envíos. Referencia S1: {COMMIT_S1[:7]}.")

## 2. Reproducir los resultados propios de S1

Se reconstruyen las medias y sus IC t de Student con los mismos datos y fórmulas de `src/inferencia.py`. Se comprueban contra las salidas publicadas en S1; la tolerancia de 0,001 corresponde a su redondeo a tres decimales.

También se recuperan el chi-cuadrado, la comparación de Welch y las tres correlaciones de la matriz de S1. **La simulación utilizará las medias, desviaciones y tamaños por grupo de S1 con precisión completa**, antes del redondeo de las tablas.

In [ ]:
from src.s2_alineacion import reconstruir_s1

base_s1 = reconstruir_s1(df, VARIABLES, OBJETIVO, alfa=ALFA)
base_medias, grupos_s1 = base_s1.medias, base_s1.grupos
peso_tiempo, peso_fuera = base_s1.peso_tiempo, base_s1.peso_fuera
delta_s1, ee_delta, ic_delta_s1 = base_s1.delta, base_s1.ee_delta, base_s1.ic_delta
welch = base_s1.welch
tabla_contingencia = base_s1.tabla_contingencia
chi_s1, p_chi_s1, gl_chi = base_s1.chi, base_s1.p_chi, base_s1.gl_chi
esperados, v_cramer = base_s1.esperados, base_s1.v_cramer
PARES, base_correlaciones = base_s1.pares, base_s1.correlaciones

# Equivalencia con los resultados publicados en Sumativa 1
np.testing.assert_allclose(base_medias[["media","ic_t_inf","ic_t_sup"]],
    [[3633.880,3601.646,3666.115],[210.239,209.293,211.186],
     [13.373,13.070,13.676]], atol=0.001,rtol=0)
np.testing.assert_allclose(grupos_s1[["n","media","de"]],
    [[3992,4166.35,1574.66],[5907,3274.03,1577.85]],atol=0.006,rtol=0)
np.testing.assert_allclose([delta_s1,*ic_delta_s1],
    [-892.316,-955.611,-829.021],atol=0.001,rtol=0)
np.testing.assert_allclose([chi_s1,p_chi_s1],[0.743,0.690],atol=0.001,rtol=0)
np.testing.assert_allclose(base_correlaciones.rho_s1,[-0.318,0.293,-0.010],atol=0.001,rtol=0)
display(base_medias.rename(index=NOMBRES).round(4))
display(grupos_s1[["n","media","de"]].round(4))
display(base_correlaciones.round(4))
print(f"S1 verificada: chi² = {chi_s1:.4f}; p = {p_chi_s1:.4f}; V = {v_cramer:.4f}.")
print(f"Welch: Δ = {delta_s1:.3f} g; IC95% = {ic_delta_s1.round(3)}; p = {welch.pvalue:.3g}.")

## 3. Bootstrap: ¿son confiables los intervalos de las tres medias?

**Método.** Para cada variable se toman 10.000 muestras con reemplazo, cada una del mismo tamaño que sus observaciones disponibles. Se calcula una media por muestra.

- **Percentil:** usa los percentiles 2,5 y 97,5 de las medias remuestreadas.
- **BCa:** ajusta por sesgo y aceleración; SciPy calcula la corrección usando jackknife.
- **Clásico de S1:** intervalo t de Student. Se contrastan límites, anchura y error estándar.

Los dos IC bootstrap usan **las mismas 10.000 remuestras**, evitando confundir diferencias de método con nuevas simulaciones. La distribución graficada es la de las **medias**, no la de los envíos individuales.

**Supuestos.** Los tres métodos requieren observaciones suficientemente independientes y una muestra adecuada para la población de interés. Bootstrap no exige normalidad de los datos, pero tampoco corrige sesgos de selección. La aproximación t mejora con muestras grandes y varianza finita; BCa puede ajustar asimetría, aunque no siempre es superior. El análisis se condiciona a los valores disponibles bajo el mecanismo MCAR de S1.

In [ ]:
from src.s2_bootstrap import analizar_medias

distribuciones_medias, bootstrap_medias, tabla_ic = analizar_medias(
    df, VARIABLES, base_medias, SEMILLAS["medias"], B=B, LOTE=LOTE
)
display(tabla_ic.rename(index=NOMBRES))
display(bootstrap_medias[["ancho_t", "ancho_percentil", "ancho_bca", "sesgo_bootstrap",
                         "cambio_ancho_bca_pct", "desplazamiento_limites_en_ee"]].rename(index=NOMBRES).round(4))

In [ ]:
from src.s2_visualizaciones import graficar_bootstrap
graficar_bootstrap(VARIABLES, bootstrap_medias, distribuciones_medias, NOMBRES)
for variable, r in bootstrap_medias.iterrows():
    display(Markdown(f"**{NOMBRES[variable]}.** La media es **{r.media:.3f}**. "
        f"El ancho BCa cambia **{r.cambio_ancho_bca_pct:+.2f}%** frente al clásico; "
        f"el mayor desplazamiento de sus límites equivale a **{r.desplazamiento_limites_en_ee:.3f}** errores estándar. "
        f"El sesgo bootstrap estimado es **{r.sesgo_bootstrap:.4f}**. "
        f"Asimetría de los datos: {r.asimetria_datos:.2f}; de las medias remuestreadas: {r.asimetria_medias_bootstrap:.2f}."))

**Lectura sencilla.** La proximidad de los intervalos respalda la precisión calculada en S1 bajo los mismos supuestos. Aunque los descuentos individuales son asimétricos, promediar muchos envíos hace más simétrica la distribución de su media. Por eso puede existir acuerdo entre t, percentil y BCa.

Este acuerdo no demuestra que el promedio del descuento sea resistente a retirar valores altos: eso se evalúa en la sección 7. Un IC del 95 % expresa la cobertura de un procedimiento en muestreos repetidos; no contiene necesariamente al 95 % de los productos ni garantiza representatividad.

## 4. Prueba de permutación: 

### 4.1. Asociación entre modo de transporte y cumplimiento del plazo de entrega

Se realiza una prueba de permutación para evaluar la asociación entre el modo de transporte y el cumplimiento del plazo de entrega, utilizando como estadístico de contraste el chi-cuadrado de Pearson. Este procedimiento permite complementar y verificar los resultados obtenidos en la fase S1 mediante una aproximación empírica a la distribución del estadístico bajo la hipótesis nula.

Hipótesis del contraste:

**H₀:** El modo de transporte y el cumplimiento del plazo de entrega son estadísticamente independientes.

**H₁:** Existe una asociación estadística entre el modo de transporte y el cumplimiento del plazo de entrega.

Procedimiento:

Se calcula inicialmente el estadístico chi-cuadrado de Pearson utilizando la tabla de contingencia observada. Posteriormente, se realizan 10.000 permutaciones aleatorias de las etiquetas correspondientes al cumplimiento del plazo de entrega, manteniendo fijos los modos de transporte y conservando las frecuencias marginales de ambas variables.

En cada permutación se reconstruye la tabla de contingencia y se calcula nuevamente el estadístico chi-cuadrado. De esta manera, se obtiene una distribución nula empírica que representa los valores del estadístico esperables bajo la hipótesis de independencia, suponiendo que las observaciones son independientes y que las etiquetas de cumplimiento son intercambiables bajo dicha hipótesis.

El valor p de permutación se estima mediante:

$$p_{\text{perm}}=
\frac{1+\sum_{b=1}^{B}I(\chi_b^2\geq\chi_{\text{obs}}^2)}{B+1}
$$

Donde:

$B=10.000$ corresponde al número de permutaciones realizadas.

$\chi_{\text{obs}}^2$ es el estadístico calculado con los datos originales.

$\chi_b^2$ es el estadístico obtenido en la permutación (b).

$I(\cdot)$ es una función indicadora que toma el valor 1 cuando se cumple la condición y 0 en caso contrario.

La incorporación de una unidad en el numerador y el denominador evita obtener un valor p estimado igual a cero cuando ninguna de las permutaciones genera un estadístico tan extremo como el observado.

Adicionalmente, se estima el error estándar de Monte Carlo del valor p mediante:

$$
EE_{\text{MC}}\approx
\sqrt{\frac{\hat p(1-\hat p)}{B}}
$$

Este indicador cuantifica la variabilidad numérica asociada al uso de un número finito de permutaciones. No representa la incertidumbre sobre la probabilidad de que la hipótesis nula sea verdadera.

Finalmente, el valor p obtenido mediante permutación se compara con el de la prueba chi-cuadrado de Pearson de S1, con el propósito de evaluar la consistencia de las conclusiones estadísticas obtenidas mediante ambos procedimientos.

In [ ]:
from src.s2_permutaciones import permutar_transporte
chi_nulos, permutacion = permutar_transporte(
    df, tabla_contingencia, chi_s1, p_chi_s1, gl_chi, esperados, v_cramer,
    objetivo=OBJETIVO, B=B, semilla=SEMILLAS["permutacion"], alfa=ALFA
)
p_perm = permutacion["p_permutacion"]
ee_p_perm = permutacion["ee_monte_carlo_p"]
display(pd.DataFrame([permutacion]).T.rename(columns={0: "Resultado"}))

fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
ax.hist(chi_nulos, bins=45, color=AZUL, alpha=0.8)
ax.axvline(chi_s1, color=NARANJA, lw=2, label=f"Observado = {chi_s1:.3f}")
ax.set(title="Distribución bajo H₀: 10.000 permutaciones", xlabel="Chi-cuadrado", ylabel="Frecuencia")
ax.legend()
plt.show()
plt.close(fig)

display(Markdown(f"**Resultado.** S1 obtuvo p = **{p_chi_s1:.4f}** y la permutación obtiene "
    f"p = **{p_perm:.4f}**. Su diferencia es {abs(p_perm-p_chi_s1):.4f}; "
    f"el margen Monte Carlo aproximado del p es ±{1.96*ee_p_perm:.4f}. "
    f"Ambos métodos {'coinciden' if (p_perm < ALFA) == (p_chi_s1 < ALFA) else 'difieren'} "
    f"en la decisión con α = 0,05. El menor conteo esperado es {esperados.min():.1f}."))

**Conclusión sencilla.** No se detecta asociación estadísticamente significativa entre transporte y puntualidad en estos datos. Esto **no demuestra que los transportes sean equivalentes o independientes**; significa que la evidencia disponible no permite rechazar H₀.

Los conteos esperados son altos, por lo que la aproximación chi-cuadrado de S1 es adecuada y eficiente. La permutación ofrece una comprobación con los márgenes observados; aquí no hay un motivo fuerte para preferirla como sustituto. Ambas dependen de la independencia entre envíos, que no puede verificarse con los campos disponibles. El V de Cramér pequeño complementa el p: la asociación observada también es pequeña. No se justifica recomendar un cambio de transporte solo con este contraste.


### 4.2. Diferencia de peso promedio según cumplimiento del plazo de entrega

Se realiza una prueba de permutación para evaluar si existen diferencias estadísticamente significativas en el peso promedio de los productos entre los envíos realizados a tiempo y aquellos registrados fuera de plazo. Este procedimiento complementa el análisis desarrollado en la fase S1 y permite contrastar la diferencia observada mediante una distribución nula generada empíricamente.

Hipótesis del contraste:

**H₀:** El peso promedio poblacional de los productos es igual entre los envíos realizados a tiempo y aquellos registrados fuera de plazo.

**H₁:** El peso promedio poblacional de los productos difiere entre ambos grupos.

En términos matemáticos:

$$
H_0:\mu_{\text{fuera}}-\mu_{\text{tiempo}}=0
$$

$$
H_1:\mu_{\text{fuera}}-\mu_{\text{tiempo}}\neq0
$$

Procedimiento:

Inicialmente, se calcula la diferencia observada entre las medias muestrales de ambos grupos, utilizando como estadístico de contraste:

$$
\Delta_{\text{obs}}=\bar X_{\text{fuera}}-\bar X_{\text{tiempo}}
$$

Posteriormente, se realizan $(B)$ permutaciones aleatorias de las observaciones, reasignando los pesos entre ambos grupos sin reemplazo y conservando sus tamaños muestrales originales. En cada permutación se calcula nuevamente la diferencia de medias.

Este procedimiento genera una distribución nula empírica que representa las diferencias de medias esperables cuando las etiquetas de cumplimiento son intercambiables respecto del peso de los productos, suponiendo independencia entre las observaciones.

Bajo esta hipótesis, la distribución de las diferencias obtenidas mediante permutación se encuentra centrada alrededor de cero.

El valor p bilateral de permutación se estima mediante:

$$
p_{\text{perm}}=
\frac{1+\sum_{b=1}^{B}I\left(|\Delta_b^*|\geq|\Delta_{\text{obs}}|\right)}{B+1}
$$

Donde:

$B$ corresponde al número de permutaciones realizadas.

$\Delta_{\text{obs}}$ representa la diferencia de medias observada en los datos originales.

$\Delta_b^*$ corresponde a la diferencia de medias obtenida en la permutación (b).

$I(\cdot)$ es una función indicadora que toma el valor 1 cuando la diferencia permutada es igual o más extrema que la observada, en valor absoluto.

La utilización de valores absolutos permite desarrollar un contraste bilateral, considerando diferencias en ambas direcciones. Asimismo, la incorporación de una unidad en el numerador y el denominador evita obtener un valor p estimado igual a cero cuando ninguna permutación genera una diferencia tan extrema como la observada.

Contraste complementario mediante la prueba t de Welch:

Adicionalmente, se aplica una prueba t de Welch para comparar las medias de ambos grupos sin asumir igualdad de varianzas poblacionales.

El estadístico se calcula mediante:

$$
t=
\frac{\bar X_{\text{fuera}}-\bar X_{\text{tiempo}}}
{\sqrt{\frac{s_{\text{fuera}}^2}{n_{\text{fuera}}}+
\frac{s_{\text{tiempo}}^2}{n_{\text{tiempo}}}}}
$$

Donde $s^2$ representa la varianza muestral y $n$ el tamaño de cada grupo.

Finalmente, se comparan los valores p obtenidos mediante la prueba de permutación y la prueba t de Welch, con el propósito de evaluar la consistencia de las conclusiones estadísticas sobre las diferencias en el peso promedio de los productos según el cumplimiento del plazo de entrega.


In [ ]:
# Permutación bilateral independiente: valida el contraste Welch de S1.
# La elección aleatoria sin reemplazo equivale a permutar las etiquetas.
from src.validacion_s2 import permutacion_diferencia_medias

resultado_prueba = permutacion_diferencia_medias(
    df, variable="Weight_in_gms", objetivo=OBJETIVO,
    iteraciones=B, semilla=SEMILLAS["permutacion_peso"], alfa=ALFA,
)
reps = resultado_prueba["replicas"]
resumen_permutacion_peso = {
    "n_total": resultado_prueba["n"],
    "n_a_tiempo": resultado_prueba["n_a_tiempo"],
    "n_fuera_plazo": resultado_prueba["n_fuera_plazo"],
    "diferencia_observada_g": resultado_prueba["diferencia"],
    "intervalo_central_nulo_95_g": resultado_prueba["intervalo_central_nulo_95"],
    "rango_nulo_simulado_g": resultado_prueba["rango_nulo_simulado"],
    "p_permutacion": resultado_prueba["p_permutacion"],
    "p_welch_s1": resultado_prueba["p_welch"],
    "extremos": resultado_prueba["extremos"],
    "permutaciones": B,
    "semilla": SEMILLAS["permutacion_peso"],
    "rechaza_welch_s1": resultado_prueba["rechaza_welch_s1"],
    "rechaza_permutacion": resultado_prueba["rechaza_permutacion"],
    "supuesto_perm": "Intercambiabilidad bajo H0; no equivale automáticamente a Welch con varianzas distintas",
}
resumen_tabla = {k: (f"{v:.3e}" if k.startswith("p_") else v)
                for k, v in resumen_permutacion_peso.items()}
display(pd.Series(resumen_tabla, name="Prueba peso-permutación").to_frame())

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.hist(reps, bins=45, color=AZUL, alpha=0.8, label="Distribución nula")
ax.axvline(resultado_prueba["diferencia"], color=NARANJA, lw=2,
           label=f"Diferencia S1 = {resultado_prueba['diferencia']:.2f} g")
ax.axvline(-resultado_prueba["diferencia"], color=NARANJA, ls="--", alpha=.8,
           label="Umbral bilateral opuesto")
ax.set(xlabel="Diferencia de medias: fuera de plazo - a tiempo (g)",
       ylabel="Frecuencia", title="Permutación bilateral de la diferencia de peso (B = 10.000)")
ax.legend(fontsize=8)
plt.show(); plt.close(fig)

display(Markdown(
    f"**Interpretación basada en resultados.** Diferencia observada = "
    f"**{resultado_prueba['diferencia']:.2f} g**; p de permutación = "
    f"**{resultado_prueba['p_permutacion']:.4f}** (resolución 1/{B+1}); "
    f"p Welch S1 = **{resultado_prueba['p_welch']:.2e}**. "
    f"Ambos {'rechazan' if resultado_prueba['rechaza_welch_s1'] and resultado_prueba['rechaza_permutacion'] else 'no comparten necesariamente'} "
    f"la hipótesis de igualdad de medias al 5%. "
    f"La permutación requiere intercambiabilidad bajo H0; Welch no requiere varianzas iguales. "
    f"La evidencia es asociativa, no causal."
))


**Comparación crítica de Welch y permutación.** La diferencia de S1 se define como media de peso **fuera de plazo menos a tiempo**; un signo negativo indica menor peso medio en el primer grupo. El contraste bilateral por permutación tiene resolución numérica mínima de $1/(B+1)$, por lo que un valor de aproximadamente $0,0001$ **no equivale** al diminuto valor p paramétrico de Welch. Ambos sustentan la misma decisión al 5 % en estos datos, aunque no descansan sobre exactamente los mismos supuestos: la permutación simple de etiquetas requiere intercambiabilidad bajo la nula y Welch permite heterogeneidad de varianzas. Con muestras grandes y desviaciones similares, su concordancia resulta razonable. El intervalo central nulo dibuja la variación esperable bajo la nula y **no es** un intervalo de confianza de la diferencia real. Ni la asociación ni la significancia estadística demuestran que el peso cause atrasos.


## 5. ¿Qué tan estables son las correlaciones de S1?

Se seleccionan tres relaciones de la matriz de Spearman de S1 para estudiar distintos patrones:

| Relación | Motivo de selección |
|---|---|
| Peso y descuento | Asociación negativa; permite estudiar sensibilidad a descuentos altos. |
| Llamadas y costo | Asociación positiva entre características del envío y atención al cliente. |
| Valoración y peso | Asociación cercana a cero; permite examinar si un efecto pequeño es preciso. |

Se remuestrean **pares completos** 10.000 veces por relación y se calculan IC percentiles del 95 %. Se vuelven a calcular los rangos de Spearman **dentro de cada remuestra**, incluyendo el tratamiento de empates; no se remuestrean rangos fijados de antemano.

Un IC que excluye cero respalda la estabilidad del signo. Un IC estrecho que incluye cero puede reflejar una asociación pequeña bien delimitada: **no significa necesariamente alta variabilidad**. Se reportan anchura y error estándar sin imponer un umbral universal de estabilidad. Los IC son individuales, exploratorios y sin ajuste conjunto por multiplicidad.

In [ ]:
from src.s2_correlaciones import analizar_correlaciones

distribuciones_rho, correlaciones, tabla_correlaciones = analizar_correlaciones(
    df, PARES, SEMILLAS["correlaciones"], B=B, LOTE=LOTE
)
display(tabla_correlaciones.round(4))

from src.s2_visualizaciones import graficar_correlaciones
graficar_correlaciones(correlaciones, distribuciones_rho, AZUL, VERDE, NARANJA)
for nombre, r in correlaciones.iterrows():
    display(Markdown(f"**{nombre}:** ρ = **{r.rho_s1:.3f}**, IC95% "
        f"**[{r.ic_inf:.3f}; {r.ic_sup:.3f}]**, anchura {r.ancho_ic:.3f}. "
        f"El intervalo {'incluye' if r.incluye_cero else 'excluye'} cero: **{r.signo.lower()}**."))

**Conclusión sencilla.** Peso–descuento mantiene un signo negativo y llamadas–costo, uno positivo. Son asociaciones de magnitud limitada, no relaciones deterministas. Valoración–peso permanece cerca de cero: su signo no queda resuelto, pero el intervalo estrecho descarta asociaciones monotónicas grandes bajo estos supuestos.

No se identifica alta variabilidad en estas tres estimaciones. No corresponde inventar una correlación inestable para completar la clasificación. La estabilidad frente al remuestreo tampoco garantiza estabilidad frente a cambios en la composición del conjunto; peso–descuento se revisa nuevamente en la sección 7.

In [ ]:
from src.s2_visualizaciones import graficar_ic_correlaciones
graficar_ic_correlaciones(correlaciones, AZUL, NARANJA)


**Comparación de estabilidad de las correlaciones**

El gráfico comparativo de los intervalos de confianza bootstrap al 95 % permite evaluar conjuntamente la estabilidad de las tres correlaciones de Spearman seleccionadas de la Sumativa 1, utilizando 10.000 remuestras por relación.

Los resultados muestran que las asociaciones entre peso y descuento, y entre llamadas de atención y costo del producto, mantienen sus signos negativo y positivo, respectivamente, dado que sus intervalos de confianza no incluyen el valor cero. Esto respalda la consistencia de la dirección de ambas asociaciones frente al remuestreo, aunque sus magnitudes son limitadas y no permiten establecer relaciones causales.

Por su parte, la correlación entre valoración del cliente y peso del producto presenta un intervalo que incluye cero. Este resultado indica que no existe evidencia suficiente para confirmar la dirección de una asociación monotónica entre ambas variables. Sin embargo, la reducida amplitud del intervalo sugiere que esta incertidumbre no se debe necesariamente a una elevada variabilidad del estimador, sino a la pequeña magnitud de la asociación observada.

En conjunto, los resultados respaldan la estabilidad del signo de dos de las tres correlaciones analizadas. No obstante, esta estabilidad debe interpretarse exclusivamente en el contexto del remuestreo bootstrap, sin asumir que las asociaciones permanecerán invariantes frente a cambios en la composición de los datos, presencia de valores extremos u otros supuestos metodológicos. Estas consideraciones se profundizan en el análisis de robustez de la sección 7.

## 6. Monte Carlo: repetir un estudio con los parámetros de S1

**Pregunta.** Si se repitiera un estudio del mismo tamaño, bajo un modelo basado en S1, ¿cuánto variaría la diferencia de peso medio entre entregas fuera de plazo y puntuales?

Se define $\Delta=\bar X_{fuera}-\bar X_{tiempo}$. Un resultado negativo significa que los envíos fuera de plazo pesan menos **en promedio**.

**Diseño.** Se generan pesos positivos con una distribución lognormal por grupo. En cada una de 10.000 iteraciones se simulan 3.992 pesos de entregas puntuales y 5.907 de entregas fuera de plazo, y se calcula la diferencia de medias. Las únicas entradas estimadas son **n, media y desviación estándar recuperadas de S1**. La distribución elegida es un supuesto de S2, no un resultado de S1.

Para una media $m$ y desviación $s$ de S1, los parámetros en escala logarítmica se obtienen por momentos:

$$\sigma_{\log}^2=\log(1+s^2/m^2),\qquad \mu_{\log}=\log(m)-\sigma_{\log}^2/2.$$

La lognormal evita pesos negativos y permite asimetría. No se afirma que describa perfectamente los pesos observados: puede generar valores fuera del rango de S1. En la sección 7 se compara con Gamma y con bootstrap. El foco es la **diferencia de medias**, no la predicción de pesos extremos.

**Convergencia.** Se revisan 1.000, 2.500, 5.000 y 10.000 iteraciones. Se distingue la DE de las diferencias (variación entre estudios) del error Monte Carlo de su media, $DE/\sqrt{M}$ (precisión numérica). Como controles operativos, se exige error numérico menor al 2 % del error estándar de S1, discrepancia con la media teórica menor a tres errores Monte Carlo y cambio de percentiles entre 5.000 y 10.000 menor al 10 % del error estándar de S1. Son tolerancias numéricas declaradas, no pruebas de validez del modelo.

In [ ]:
from src.s2_montecarlo import SimuladorMonteCarlo, resumir_distribucion

simulador = SimuladorMonteCarlo(grupos_s1, iteraciones=M, lote=LOTE, alfa=ALFA)
simular_diferencias = simulador.simular  # Igual interfaz de cálculo, implementación en src/.

mc_lognormal, cobertura_lognormal = simular_diferencias(
    "lognormal", SEMILLAS["lognormal"], devolver_ic=True
)
resumen_mc = resumir_distribucion(mc_lognormal)
display(pd.DataFrame([resumen_mc], index=["Monte Carlo lognormal"]).round(4))
print(f"Referencia teórica del modelo: media = {delta_s1:.3f} g; DE = {ee_delta:.3f} g.")
print(f"IC95% Welch de S1: [{ic_delta_s1[0]:.3f}; {ic_delta_s1[1]:.3f}] g.")

cortes = [1_000, 2_500, 5_000, M]
convergencia = pd.DataFrame([resumir_distribucion(mc_lognormal[:k]) for k in cortes])
convergencia["desvio_media_teorica"] = convergencia.media - delta_s1
convergencia["cambio_max_percentiles"] = convergencia[["p2_5", "p97_5"]].diff().abs().max(axis=1)
display(convergencia.round(4))
cambio_percentiles = float(convergencia.iloc[-1].cambio_max_percentiles)
controles_convergencia = {
    "Error MC < 2% del error estándar S1": resumen_mc["error_mc_media"] < 0.02 * ee_delta,
    "Media simulada a menos de 3 errores MC de la teórica": abs(resumen_mc["media"]-delta_s1) < 3*resumen_mc["error_mc_media"],
    "Cambio de percentiles < 10% del error estándar S1": cambio_percentiles < 0.10 * ee_delta,
}
display(pd.Series(controles_convergencia, name="Control cumplido").to_frame())

from src.s2_visualizaciones import graficar_montecarlo_convergencia
graficar_montecarlo_convergencia(mc_lognormal, delta_s1, ic_delta_s1, M, AZUL, NARANJA)
display(Markdown(f"**Lectura del resultado.** La diferencia simulada promedio es **{resumen_mc['media']:.2f} g**; "
    f"el 95 % central de las diferencias va de **{resumen_mc['p2_5']:.2f} a {resumen_mc['p97_5']:.2f} g**. "
    f"La variación entre estudios (DE) es **{resumen_mc['de']:.2f} g**, y el error numérico de su media es "
    f"**{resumen_mc['error_mc_media']:.3f} g**. Los percentiles cambian como máximo **{cambio_percentiles:.3f} g** "
    f"entre 5.000 y 10.000 iteraciones. "
    f"{'Los tres controles numéricos se cumplen.' if all(controles_convergencia.values()) else 'Revisar convergencia y aumentar M antes de entregar.'}"))

**Conclusión sencilla.** Bajo el modelo calibrado, la diferencia de peso conserva la dirección observada en S1 y su variación es pequeña frente a una brecha cercana a 892 g. Esto ayuda a dimensionar cuánto cambiaría el resultado al repetir un estudio comparable.

El acuerdo con S1 es parcialmente esperado: la simulación se calibra con sus propias medias y desviaciones. Los percentiles simulados son un **intervalo de variación bajo el modelo**; no son una nueva prueba de que el modelo sea verdadero, ni una evaluación de cobertura del IC de S1. Tampoco estiman el efecto de cambiar el peso sobre la puntualidad: los grupos están definidos por un estado de entrega observado.

### 6.1. Diagnóstico de adecuación de las familias simuladas

**Qué prueba este diagnóstico.** La convergencia de Monte Carlo acredita precisión numérica, pero no demuestra que una distribución lognormal sea un buen modelo de los pesos. Se comparan los cuantiles empíricos de S1 con cuantiles teóricos de **lognormal y Gamma calibradas exclusivamente con las medias y desviaciones de S1**. El contraste es descriptivo; no se reestiman los parámetros con una optimización de ajuste. Las discrepancias en colas se documentan antes de interpretar los escenarios.


In [ ]:
from src.validacion_s2 import diagnostico_cuantiles_peso

diagnostico_distribuciones = diagnostico_cuantiles_peso(
    peso_tiempo, peso_fuera, grupos_s1,
    probabilidades=(.05, .25, .50, .75, .95)
)
display(diagnostico_distribuciones.round(2))
from src.s2_visualizaciones import graficar_cuantiles
graficar_cuantiles(diagnostico_distribuciones, AZUL, NARANJA)
error_ajuste = diagnostico_distribuciones.groupby("grupo")[[
    "error_abs_lognormal_g", "error_abs_gamma_g"
]].mean()
display(error_ajuste.rename(columns={"error_abs_lognormal_g": "Error medio absoluto lognormal (g)",
                                      "error_abs_gamma_g": "Error medio absoluto Gamma (g)"}).round(2))
display(Markdown("**Alcance.** Los modelos pueden reproducir la media y la DE de S1 por construcción "
                 "sin reproducir todos los cuantiles observados. Diferencias grandes en estos gráficos "
                 "se deben informar como limitación de las colas; no cambian por sí solas las medias de entrada."))


### 6.2. Evaluación de la cobertura de Welch mediante Monte Carlo

**Pregunta adicional.** ¿Los intervalos de confianza de Welch al 95 % cubrirían el parámetro verdadero en aproximadamente el 95 % de estudios repetidos, **si las poblaciones generadoras siguieran los modelos lognormal o Gamma** calibrados con S1?

1. En cada uno de **10.000 estudios por distribución** se generan, de manera independiente, 3.992 observaciones para el grupo «A tiempo» y 5.907 para «Fuera de plazo», utilizando exclusivamente sus tamaños, medias y desviaciones de S1.
2. Se calcula la diferencia de medias **y el IC de Welch de cada estudio usando las varianzas de esa muestra sintética**.
3. Se registra si ese intervalo contiene el valor poblacional fijado por el modelo, `delta_modelo = media_fuera_S1 - media_tiempo_S1`. Su frecuencia es la **cobertura empírica**. También se calcula la amplitud media de los intervalos y la incertidumbre Monte Carlo de la cobertura.
4. Se compara con el **95 % nominal**, sin afirmar que eso prueba que la población real sea lognormal o Gamma.

La cobertura es una propiedad del **procedimiento de inferencia bajo un modelo de generación especificado**, no una probabilidad de que el intervalo concreto de S1 contenga su parámetro. Una cobertura próxima al 95 % respaldaría el desempeño condicional de Welch, pero **no confirma la representatividad de los datos, la independencia de los envíos ni una relación causal entre peso y atraso**.

Los escenarios lognormal y Gamma se eligen por admitir pesos positivos; las diferencias de cuantiles observadas en 6.1 impiden tratarlos como modelos logísticos validados. Para estudiar una población con forma empírica diferente, la sección 7.3 complementa con bootstrap independiente por grupo.

In [ ]:
# Reutiliza las 10.000 muestras lognormales generadas en la sección 6,
# ahora con la cobertura de sus IC Welch ya calculada en el mismo lote.
# La Gamma se calcula una sola vez y luego se reutiliza en robustez (sección 7.3).
mc_gamma, cobertura_gamma = simular_diferencias(
    "gamma", SEMILLAS["gamma"], devolver_ic=True
)

cobertura_welch_mc = pd.DataFrame([cobertura_lognormal, cobertura_gamma])
display(cobertura_welch_mc[[
    "familia", "iteraciones", "cobertura_ic_welch_95_pct",
    "error_montecarlo_cobertura_pp", "ancho_medio_ic_welch_g",
    "porcentaje_ic_excluye_cero"
]].round(3))

from src.s2_visualizaciones import graficar_cobertura
graficar_cobertura(cobertura_welch_mc, AZUL, NARANJA)
y = cobertura_welch_mc.cobertura_ic_welch_95_pct.to_numpy()
# Las cotas ±1,96 EE describen incertidumbre Monte Carlo, no el IC sobre
# una población real. La tolerancia 2 pp se usa solo como control numérico.
control_cobertura = bool(
    (cobertura_welch_mc.iteraciones >= 10_000).all()
    and np.isfinite(cobertura_welch_mc.select_dtypes(include=["number"])).all().all()
    and ((cobertura_welch_mc.cobertura_ic_welch_95_pct - 95).abs() <= 2).all()
)
display(Markdown(
    "**Interpretación crítica.** En los escenarios generadores calibrados con S1, "
    f"la cobertura fue **{y[0]:.2f}% (lognormal)** y **{y[1]:.2f}% (Gamma)**. "
    "Estos resultados son compatibles con el 95% nominal al considerar la incertidumbre "
    "de simulación. La lognormal y Gamma pueden conservar medias y varianzas por construcción "
    "sin ajustar los cuantiles empíricos: es una validación **condicional**, no una prueba "
    "de que cambiar el peso modifique la puntualidad."
))

## 7. Robustez: ¿qué cambia al retirar observaciones o variar supuestos?

### 7.1. Influencia de un envío: jackknife de las tres medias

Se calcula cuánto cambia cada media al omitir **una observación a la vez**. Para la media no es necesario reconstruir miles de tablas:

$$\bar x_{(-i)}=\frac{n\bar x-x_i}{n-1},\qquad cambio_i=\bar x_{(-i)}-\bar x.$$

Se muestran los tres ID de mayor cambio absoluto por variable. Se informa el cambio en sus unidades y como fracción del error estándar de S1; no existe aquí un umbral estadístico universal para declarar una observación problemática. En caso de empate se ordena por ID. Ser el caso más influyente de esta lista no significa ser un error.

In [ ]:
from src.s2_robustez import analizar_jackknife
influencia, resumen_influencia = analizar_jackknife(df, VARIABLES, base_medias)
display(influencia.round(6))
display(resumen_influencia.rename(index=NOMBRES).round(6))
max_influencia_ee = resumen_influencia.max_cambio_en_ee.max()
display(Markdown(f"**Conclusión.** El mayor cambio individual equivale a **{max_influencia_ee:.4f}** "
    "errores estándar de S1. Ninguno de los casos listados domina las medias: el efecto individual es pequeño "
    "frente a la incertidumbre de estimación. Esto no descarta una influencia importante de muchos valores altos juntos."))

### 7.2. Sensibilidad colectiva a valores extremos

Se identifican valores fuera de $[Q_1-1,5RIC;\ Q_3+1,5RIC]$, usando la misma regla exploratoria de S1. Se auditan las tres variables. Para descuento se compara el análisis completo con un escenario que excluye esos casos, recalculando su media, IC bootstrap y correlación con peso.

**La exclusión es solo un escenario de sensibilidad.** Al retirar una parte importante de la muestra, cambia la población descrita. No se sobrescribe `df` ni se utiliza este escenario como base principal.

In [ ]:
from src.s2_robustez import analizar_extremos

(auditoria_extremos, sensibilidad_extremos, n_excluidos, pct_desc_filtrado,
 media_desc_filtrado, rho_filtrado) = analizar_extremos(
    df, VARIABLES, bootstrap_medias, correlaciones, SEMILLAS, B=B, LOTE=LOTE
)
desc_base = bootstrap_medias.loc["Discount_offered"]
rho_base = correlaciones.loc["Peso y descuento"]
display(auditoria_extremos.rename(index=NOMBRES).round(3))
display(sensibilidad_extremos.round(4))


from src.s2_visualizaciones import graficar_extremos
graficar_extremos(sensibilidad_extremos, AZUL, NARANJA)
cambio_desc_pct = 100*(media_desc_filtrado/desc_base.media-1)
display(Markdown(f"**Conclusión.** La regla marca **{n_excluidos:,} envíos ({100*n_excluidos/len(df):.2f}%)**. "
    f"Al retirarlos, la media del descuento pasa de **{desc_base.media:.3f} a {media_desc_filtrado:.3f}** "
    f"({cambio_desc_pct:+.2f}%). Spearman peso–descuento pasa de **{rho_base.rho_s1:.3f} a {rho_filtrado:.3f}**. "
    "La media y la magnitud de esta asociación son sensibles a la composición de los descuentos. "
    "El cambio colectivo es mucho mayor que el de retirar un solo envío."))

Peso y costo no presentan casos fuera de sus cercas de 1,5 RIC: con esta regla no habría una exclusión adicional que comparar en esas variables. Esto no demuestra ausencia de valores problemáticos bajo otros criterios.

Los descuentos altos pueden ser parte legítima del negocio. Para S3 se recomienda conservarlos, revisar su significado y complementar la media con mediana, cuantiles y análisis por grupos de descuento. La asociación peso–descuento mantiene un signo negativo en el escenario, pero pierde buena parte de su magnitud. Por tanto, **signo estable no equivale a magnitud robusta**.

### 7.3. Sensibilidad a la distribución supuesta

Se compara el escenario lognormal con una **Gamma** que conserva exactamente las mismas medias y desviaciones de S1: forma $k=(m/s)^2$ y escala $\theta=s^2/m$. Cada escenario utiliza 10.000 estudios simulados completos.

Como contraste adicional, se hace bootstrap de la diferencia de medias, remuestreando **por separado dentro de cada estado de entrega** y manteniendo los tamaños. Así no se impone una familia de distribución para los pesos.

Los percentiles de Monte Carlo describen variación bajo cada modelo; el IC percentil bootstrap estima incertidumbre por remuestreo de los datos. Su comparación es una comprobación de sensibilidad, no los convierte en intervalos conceptualmente idénticos.

In [ ]:
from src.s2_robustez import comparar_familias

dist_delta_boot, comparacion_modelos = comparar_familias(
    peso_fuera, peso_tiempo, mc_lognormal, mc_gamma, delta_s1, ee_delta,
    semilla=SEMILLAS["diferencia_bootstrap"], B=B, LOTE=LOTE
)
display(comparacion_modelos.round(4))

from src.s2_visualizaciones import graficar_modelos
graficar_modelos(mc_lognormal, mc_gamma, dist_delta_boot, delta_s1, AZUL, NARANJA, VERDE)
desvio_de_max = comparacion_modelos.cambio_de_vs_s1_pct.abs().max()
display(Markdown(f"**Resultado.** La mayor diferencia relativa entre la DE calculada y el error estándar "
    f"de S1 es **{desvio_de_max:.2f}%**. Los tres intervalos centrales "
    f"{'permanecen por debajo' if (comparacion_modelos.p97_5 < 0).all() else 'no permanecen todos por debajo'} "
    "de cero. La dirección y la escala de incertidumbre de la diferencia son similares con los métodos examinados."))

**Interpretación.** La comparación respalda la diferencia de medias frente a estos cambios concretos de distribución. El acuerdo entre Lognormal y Gamma es esperable porque comparten media y varianza y los grupos son grandes. No valida la forma completa de los pesos ni su comportamiento en las colas. El bootstrap añade evidencia sin imponer esas familias, pero sigue dependiendo de la muestra observada y de la independencia de los envíos.

### 7.4. Sensibilidad a las entradas de la simulación

Para evaluar también el impacto de los parámetros, se usan como escenarios la diferencia central de S1 y los dos extremos de su IC de Welch. Se mantiene fija la media del grupo a tiempo y se define:

$$m_{fuera,escenario}=m_{tiempo,S1}+\Delta_{escenario}.$$

Los tamaños y desviaciones permanecen en los valores de S1. Se simulan 10.000 estudios lognormales por escenario. Estos extremos son **valores alternativos plausibles de la diferencia según S1**, no cambios causados por una intervención. Se analizan por separado; no se mezclan para construir un nuevo IC ni representan toda la incertidumbre conjunta de los parámetros.

In [ ]:
from src.s2_montecarlo import analizar_escenarios

sensibilidad_entradas, distribuciones_escenarios = analizar_escenarios(
    simulador, grupos_s1, ic_delta_s1, delta_s1, mc_lognormal, SEMILLAS
)
display(sensibilidad_entradas.round(3))

from src.s2_visualizaciones import graficar_escenarios
graficar_escenarios(sensibilidad_entradas, AZUL)
display(Markdown(f"**Conclusión.** Al usar diferencias de entrada entre **{ic_delta_s1[0]:.2f} y "
    f"{ic_delta_s1[1]:.2f} g**, las medias simuladas van de **{sensibilidad_entradas.media.min():.2f} "
    f"a {sensibilidad_entradas.media.max():.2f} g**. "
    f"{'Todos' if (sensibilidad_entradas.p97_5 < 0).all() else 'No todos'} los intervalos de variación quedan "
    "por debajo de cero. La magnitud se mueve con la entrada, como corresponde; la dirección se mantiene "
    "dentro del rango estudiado. No es una garantía frente a otros escenarios o sesgos de muestreo."))

### 7.5. Robustez de Welch frente a pesos extremos: medias recortadas

**Objetivo.** Welch y Monte Carlo describen diferencias de medias tradicionales. Para evaluar si la dirección depende mucho de observaciones extremas, se calcula la **media recortada 10 % en cada cola, por grupo**, y un intervalo percentil bootstrap del 95 % sobre su diferencia con 10.000 remuestras independientes dentro de cada grupo.

Es una **medida de sensibilidad complementaria**, no un reemplazo de la prueba de Welch de S1. Se muestran juntas la diferencia original, la recortada y sus intervalos; la decisión sobre robustez se basa en dirección, magnitud y ancho del intervalo, no solo en si incluye cero.


In [ ]:
from src.s2_robustez import analizar_medias_recortadas

delta_recortada, boot_recortada, ic_recortada, resumen_recortadas = analizar_medias_recortadas(
    peso_fuera, peso_tiempo, delta_s1, semilla=SEMILLAS["robustez_recortadas"], B=B, LOTE=LOTE
)
display(pd.Series(resumen_recortadas, name="Robustez Welch").to_frame())
from src.s2_visualizaciones import graficar_recortadas
graficar_recortadas(delta_s1, ic_delta_s1, delta_recortada, ic_recortada, AZUL, NARANJA)
display(Markdown(
    f"**Lectura.** La diferencia recortada es **{delta_recortada:.2f} g** "
    f"(IC percentil 95% [{ic_recortada[0]:.2f}; {ic_recortada[1]:.2f}] g), "
    f"frente a **{delta_s1:.2f} g** con Welch de S1. "
    f"La dirección {'se mantiene' if resumen_recortadas['mismo_signo'] else 'cambia'} "
    "al atenuar los extremos. No equivale a una prueba causal ni reemplaza el contraste original."
))


## 8. Conclusiones y preparación para S3

La síntesis distingue tres conceptos: **precisión** (incertidumbre de la estimación), **estabilidad** (consistencia al remuestrear) y **robustez** (sensibilidad a los cambios concretos que se examinaron). Un resultado puede ser preciso y sensible a la vez.

La versión revisada incorpora una segunda permutación (peso), diagnósticos de cuantiles y robustez por medias recortadas; todos deben guardarse en el JSON regenerado.


In [ ]:
from src.s2_resultados import crear_sintesis

sintesis = crear_sintesis()
display(sintesis)

display(Markdown(f"""
**Conclusión general.** Los métodos computacionales respaldan varios resultados de S1, pero muestran que el descuento exige cautela. La concordancia entre métodos ayuda a evaluar las estimaciones; no elimina las limitaciones del conjunto de datos.

**Conclusión técnica y estadística.** El contraste de peso por permutación (p = **{resultado_prueba['p_permutacion']:.4f}**) confirma la decisión de Welch en S1.  Las tres medias presentan IC clásicos, percentiles y BCa próximos. La prueba de transporte conserva su decisión (p de permutación = **{p_perm:.4f}**). Dos correlaciones mantienen su signo y valoración–peso permanece cerca de cero. La diferencia de peso, **{delta_s1:.2f} g**, fue además evaluada con media recortada (**{delta_recortada:.2f} g**), que sirve como análisis de sensibilidad. En cambio, excluir descuentos altos reduce su media **{abs(cambio_desc_pct):.2f}%** y debilita su correlación con peso.

**Conclusión para decisiones.** Estos resultados orientan a investigar perfiles de productos y la composición de los descuentos. No justifican afirmar que cambiar el transporte o el peso reducirá los atrasos. Antes de intervenir conviene incorporar destino, distancia, fecha y duración del retraso, además de verificar la unidad del descuento.

**Continuidad hacia S3.** Se exportan estimaciones e intervalos, correlaciones, ID con mayor influencia individual, escenarios y recomendaciones. Deben conservarse los casos altos salvo evidencia de error. Si se evalúa imputación o predicción, separar entrenamiento y evaluación; usar los valores MCAR reservados solo como verdad de evaluación, nunca para ajustar el método. La estabilidad inferencial aquí observada no garantiza desempeño predictivo.
"""))

**Limitaciones comunes.** El origen de la muestra y la independencia entre envíos no están confirmados; no se dispone de fechas ni identificadores de cliente para evaluar dependencia temporal o repetición. Los faltantes fueron generados artificialmente con MCAR y no reproducen todos los mecanismos de ausencia reales. Bootstrap y permutación actúan sobre esta muestra; no corrigen representatividad, confusión ni causalidad. Los IC y contrastes son exploratorios e individuales. La convergencia reduce el error numérico, pero no corrige un modelo inadecuado.

### 8.1. Archivo de resultados validados

La siguiente celda crea o actualiza **`data/processed/resultados_validos_s2.json`**, dentro de una carpeta que ya existe. Incluye números con precisión numérica, sin el redondeo de presentación, además de procedencia, semillas, repeticiones, influencias y recomendaciones. Este archivo y el notebook constituyen la entrada analítica para S3; el PDF de S2 se redacta a partir de estas mismas salidas.

**Verificación de modelos.** El diagnóstico por cuantiles es descriptivo; ajustes por momentos pueden coincidir en media y DE mientras difieren en colas. El análisis de medias recortadas usa otro estimador y ayuda a distinguir el signo estable de la sensibilidad de magnitudes.


In [ ]:
from src.s2_resultados import guardar_resultados

contexto_s2 = {
    "REPO_URL": REPO_URL,
    "COMMIT_S1": COMMIT_S1,
    "hash_datos": hash_datos,
    "VERSIONES": VERSIONES,
    "B": B,
    "M": M,
    "LOTE": LOTE,
    "SEMILLAS": SEMILLAS,
    "bootstrap_medias": bootstrap_medias,
    "permutacion": permutacion,
    "resumen_permutacion_peso": resumen_permutacion_peso,
    "correlaciones": correlaciones,
    "grupos_s1": grupos_s1,
    "delta_s1": delta_s1,
    "ic_delta_s1": ic_delta_s1,
    "welch": welch,
    "ee_delta": ee_delta,
    "resumen_mc": resumen_mc,
    "convergencia": convergencia,
    "controles_convergencia": controles_convergencia,
    "cobertura_welch_mc": cobertura_welch_mc,
    "control_cobertura": control_cobertura,
    "comparacion_modelos": comparacion_modelos,
    "diagnostico_distribuciones": diagnostico_distribuciones,
    "sensibilidad_entradas": sensibilidad_entradas,
    "influencia": influencia,
    "resumen_recortadas": resumen_recortadas,
    "resumen_influencia": resumen_influencia,
    "auditoria_extremos": auditoria_extremos,
    "sensibilidad_extremos": sensibilidad_extremos,
    "n_excluidos": n_excluidos,
    "df": df,
    "pct_desc_filtrado": pct_desc_filtrado,
    "sintesis": sintesis,
}
resultados_s2 = guardar_resultados(RAIZ, RUTA_SALIDA, contexto_s2)

## 9. Verificación final y referencias

Los siguientes controles verifican la evidencia calculada. No asignan una calificación automática: la interpretación y la entrega completa también se evalúan.

In [ ]:
controles = {
    "Datos y cifras de S1 verificados": hash_datos == HASH_DATOS_S1,
    "Tres medias; ≥10.000 remuestras por media": len(distribuciones_medias) == 3 and all(len(x) >= 10_000 for x in distribuciones_medias.values()),
    "IC percentil y BCa finitos y ordenados": bool(np.isfinite(bootstrap_medias[["percentil_inf", "percentil_sup", "bca_inf", "bca_sup"]]).all().all()
        and (bootstrap_medias.percentil_inf < bootstrap_medias.percentil_sup).all()
        and (bootstrap_medias.bca_inf < bootstrap_medias.bca_sup).all()),
    "Permutación: ≥10.000 y p válido": len(chi_nulos) >= 10_000 and 0 < p_perm <= 1,
    "Permutación peso: B válido, coherencia con Welch y etiqueta correcta":
        len(resultado_prueba["replicas"]) >= 10_000 and 0 < resultado_prueba["p_permutacion"] <= 1
        and resultado_prueba["rechaza_welch_s1"] == (resultado_prueba["p_welch"] < ALFA)
        and resultado_prueba["rechaza_permutacion"] == (resultado_prueba["p_permutacion"] < ALFA),
    "Tres correlaciones, pares e IC95% válidos": len(correlaciones) == 3 and bool((correlaciones.ic_inf < correlaciones.ic_sup).all())
        and all(len(x) >= 10_000 and np.isfinite(x).all() for x in distribuciones_rho.values()),
    "Monte Carlo: ≥10.000 por escenario": all(len(x) >= 10_000 and np.isfinite(x).all()
        for x in [mc_lognormal, mc_gamma, *distribuciones_escenarios.values()]),
    "Diagnóstico cuantiles Gamma/Lognormal completo": len(diagnostico_distribuciones) == 10
        and bool(np.isfinite(diagnostico_distribuciones.select_dtypes(include=["number"])).all().all()),
    "Robustez media recortada 10% y 10.000 remuestras":
        len(boot_recortada.bootstrap_distribution) >= 10_000 and np.isfinite(ic_recortada).all()
        and ic_recortada[0] < ic_recortada[1],
    "JSON incluye dos permutaciones, diagnósticos y recortadas":
        all(k in json.loads(RUTA_SALIDA.read_text(encoding="utf-8")) for k in
            ["permutacion", "permutacion_peso", "monte_carlo", "robustez"]),
    "Cobertura Monte Carlo de Welch evaluada bajo dos familias":
        bool(control_cobertura) and len(cobertura_welch_mc) == 2
        and (cobertura_welch_mc.iteraciones >= 10_000).all(),
    "Convergencia numérica principal comprobada": all(controles_convergencia.values()),
    "Robustez: influencia, extremos, familias y entradas": len(influencia) == 9
        and len(sensibilidad_extremos) == 2 and len(comparacion_modelos) == 3 and len(sensibilidad_entradas) == 3,
    "JSON válido con recomendaciones para S3": len(json.loads(RUTA_SALIDA.read_text(encoding="utf-8"))["sintesis_y_recomendaciones_s3"]) == 8,
    "CSV fuente sin cambios": hashlib.sha256(RUTA_DATOS.read_bytes().replace(b"\r\n", b"\n")).hexdigest() == HASH_DATOS_S1,
}
display(pd.DataFrame({"Control": controles.keys(), "Cumple": [bool(x) for x in controles.values()]}))
assert all(controles.values()), "Hay controles pendientes: revisar antes de entregar."
print("Todos los controles computacionales se cumplen. Guardar el notebook con estas salidas.")

### Referencias y trazabilidad

1. Universidad Andrés Bello (2026). *Sumativa 2. Informe 3: instrucciones y rúbrica, fase 3*. Documento del curso proporcionado al equipo; seis criterios, 87 puntos.
2. Grupo 7 (2026). [Repositorio de Estadística Computacional, versión S1 `0c7ffac`](https://github.com/jorge170102/ESTAD-STICA-COMPUTACIONAL---Grupo-7--MCDI501.202682.1914.EL.ON/tree/0c7ffaca4e45525ab434f6c9305128221e4ef1c4). Notebook `notebooks/sumativa_1.ipynb`, datos preparados y módulos de inferencia y análisis bivariado.
3. Grupo 7 (2026). *Informe Sumativa 1: factores asociados con la puntualidad de envíos en comercio electrónico*. Informe institucional proporcionado, 12 de octubre de 2026.
4. SciPy. [Bootstrap: percentil, BCa y remuestreo por pares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html).
5. SciPy. [Pruebas de permutación y corrección del valor p](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.permutation_test.html).
6. SciPy. [Correlación de Spearman](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html).

**Antes de entregar:** guardar este notebook con todas sus salidas, elaborar el PDF de máximo siete páginas a partir de estos resultados, incluir el JSON exportado y revisar la condición formal de carpetas S1/S2 y el enlace a GitHub. Este notebook conserva deliberadamente la estructura actual; no modifica los archivos de S1 ni publica cambios en el repositorio.